### 5. ArcPy functions

Similar to any Python function, an ArcPy **function** takes arguments, performs a specific task and returns a value. Esri's documentation groups ArcPy functions into two kinds: geoprocessing tools, which are the same tools you run from the Geoprocessing pane, and other functions that support your scripts, such as checking whether a dataset exists or whether a license is available.

#### 5a. General ArcPy functions

There are several ArcPy functions that perform a variety of specific tasks. The ArcGIS Pro Python reference provides a list of [ArcPy Functions](https://doc.esri.com/en/arcgis-pro/latest/arcpy/functions/alphabetical-list-of-arcpy-functions.html) and includes the description, syntax and code samples for each function.

`arcpy.Exists()` returns `True` if a dataset exists and `False` if it does not. Checking that an input exists before you use it catches a wrong name or path early.



In [ ]:
# Example 1 - Check for datasets in arcpy.env.workspace

# We will use the workspace set in Section 4
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# A dataset name without a path: ArcPy looks for it in the workspace
pour_points = "pourpoints.shp"
print(f"pour_points: {pour_points}")
points_exist = arcpy.Exists(pour_points)
print(f"points_exist: {points_exist} in {arcpy.env.workspace}")

# A dataset that is not in the workspace returns False
rivers_exist = arcpy.Exists("rivers.shp")
print(f"rivers_exist: {rivers_exist} in {arcpy.env.workspace}")

In [ ]:
# Example 2 - Check for datasets by providing the full path

# ArcPy uses the path exactly as written. Update this path to
# match the location of the data folder on your computer
points_path = r"C:\GIST513\data\pourpoints.shp"
print(f"points_path: {points_path}")

path_exists = arcpy.Exists(points_path)
print(f"path_exists: {path_exists}")

In [ ]:
# Example 3 - Changing the workspace

# Update this path to a folder on your computer that does not contain 
# pourpoints.shp
arcpy.env.workspace = r"C:\GIST513"
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# The dataset name is now looked up in the new workspace
print(f"arcpy.Exists(pour_points): {arcpy.Exists(pour_points)}")

# The full path does not depend on the workspace, so it still returns True
print(f"arcpy.Exists(points_path): {arcpy.Exists(points_path)}")

# Set the workspace back to the data folder for the rest of this notebook
arcpy.env.workspace = r"C:\GIST513\data"
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

In [ ]:
# Example 4 - Using conditional statements

# Because Exists returns True or False, it can be the condition of a 
# conditional statement
if arcpy.Exists(pour_points):
    print(f"\t{pour_points} exists")
else:
    print(f"\t{pour_points} was not found")

Some tools and modules require an **extension**, an add-on license for ArcGIS Pro, such as Spatial Analyst for the `arcpy.sa` module. `arcpy.CheckExtension()` returns a string that describes whether an extension license is available: `"Available"`, `"Unavailable"`, `"NotLicensed"`, or `"Failed"`.

In [ ]:
# Check whether the Spatial Analyst extension is available
spatial_status = arcpy.CheckExtension("Spatial")
print(f"spatial_status: {spatial_status}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 5a] Create a variable to store the name of a dataset in the data folder.
# Use arcpy.Exists() and if/else statements to print whether the dataset
# exists.


# 5b] Create a variable to store the name of a dataset that is NOT in the
# data folder. Repeat the check from 5a with this variable.


# 5c] Use arcpy.CheckExtension() to check an extension of your choosing,
# such as "3D" or "Network". Print the result with a descriptive message.

#### 5b. Toolboxes and tools

Geoprocessing tools are called the same way as other ArcPy functions, but they are organized differently.

In ArcGIS Pro, geoprocessing tools are grouped into **toolboxes**, such as Analysis Tools and Data Management Tools. Each toolbox has a short name called an **alias**, such as `analysis` or `management`. In ArcPy, each toolbox alias works like a module, so you run a tool with dot notation: `arcpy.<toolbox alias>.<tool name>()`. For example, the Buffer tool in the Analysis Tools toolbox is `arcpy.analysis.Buffer()`. 

The [ArcGIS Pro tool reference](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/main/arcgis-pro-tool-reference.html) lists every toolbox, its alias, and its tools. It includes a description of how each tool works, its Python syntax, and code samples.

**Example: Buffer Tool**

Buffer has three required parameters: `in_features`, `out_feature_class`, and `buffer_distance_or_field`. The buffer distance can be a **linear unit**, a string with a distance and a unit, such as `"500 Meters"`. Running a tool with positional arguments works the same way as calling a function: each argument is matched to a parameter by its position.

In [ ]:
# Run Buffer with positional arguments
# We will use variable pour_points from Section 6
arcpy.analysis.Buffer(pour_points, "pourpoints_buffer.shp", "500 Meters")

Keyword arguments name each parameter, which makes a tool call easier to read and lets you set one optional parameter without passing all the ones before it. A call can span several lines as long as the lines are inside its parentheses.

In [ ]:
# Run Buffer with keyword arguments. The optional dissolve_option parameter
# is set to "ALL" so that overlapping buffers merge into one
# feature
arcpy.analysis.Buffer(
    in_features=pour_points,
    out_feature_class="pourpoints_buffer_dissolved.shp",
    buffer_distance_or_field="500 Meters",
    dissolve_option="ALL"
)

Older code, including a lot of AI-generated code, writes the toolbox alias after the tool name with an underscore, such as `arcpy.Buffer_analysis()`. It runs the same tool. Recognize this syntax when you see it, but you do not need to write it.

In [ ]:
# Older syntax: tool name, underscore, then toolbox alias
# arcpy.Buffer_analysis(pour_points, "pourpoints_buffer.shp", "500 Meters")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 6. Tool results and messages

The tool calls shown above did not store their return value. Every tool that runs successfully returns a **Result object**, an object created from the `Result` class that stores information about the tool run, including its outputs and its messages. You store it in a variable the same way you store any function's return value. For all of its properties and methods, see the [Result class](https://doc.esri.com/en/arcgis-pro/latest/arcpy/classes/result.html) page in the ArcGIS Pro Python reference.

In [ ]:
# Store the Result object returned by Buffer
buffer_result = arcpy.analysis.Buffer(
    pour_points, "pourpoints_buffer_1km.shp", "1 Kilometers"
)
print(f"type(buffer_result): {type(buffer_result)}")

The `getOutput()` method returns an output of the tool by its index. For Buffer, `getOutput(0)` returns the path of the output feature class.

In [ ]:
# getOutput(0) returns the first output, the output feature class path
buffer_output = buffer_result.getOutput(0)
print(f"buffer_output: {buffer_output}")
print(f"arcpy.Exists(buffer_output): {arcpy.Exists(buffer_output)}")

Every tool produces **messages**, the same messages you see in the Geoprocessing pane's details: when the tool started, when it finished, and any warnings or errors. Each message has a **severity**:

* `0`: informative message
* `1`: warning message
* `2`: error message

The `getMessages()` method returns messages as one string. With no argument or with `0`, it returns all messages. With `1`, it returns only warning messages, and with `2`, only error messages. A tool that fails raises an error instead of returning a Result object, so the error messages of a successful tool run are empty. The `messageCount` property stores the number of messages.

In [ ]:
# We will use variable buffer_result from the cell above
print(f"messageCount: {buffer_result.messageCount}")

# No argument returns all messages
print(buffer_result.getMessages())

# 1 returns only warning messages. The quotes show an empty string
print(f"Warnings: '{buffer_result.getMessages(1)}'")

# 2 returns only error messages
print(f"Errors: '{buffer_result.getMessages(2)}'")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 6a] Run your tool from 7c again, and store the Result object in a new
# variable.


# 6b] Use getOutput(0) on your variable from 8a to get the tool's first
# output. Print it, and use arcpy.Exists() to check whether it exists.

In [ ]:
# 6c] Print the message count and all messages of your Result object
# from 8a.


# 6d] Write a for loop with range() and getMessage() to print each message
# of your Result object from 8a with its index.

### 7. Tool examples

This section runs several geoprocessing tools, using the syntax, keyword arguments, and Result objects from Sections 5b and 6.

Refer to the [ArcGIS Pro tool reference](https://pro.arcgis.com/en/pro-app/latest/tool-reference/main/arcgis-pro-tool-reference.htm) for more information about each tool, including its toolbox, a description of how it works, its Python syntax, and code samples.

#### 7a. Clip tool

The **Clip** tool keeps the parts of the input features that fall inside the clip features.

In [ ]:
# Example 1 - Clip a point layer with a polygon layer

# We will use variable pour_points from Section 5a
# Update this name to a polygon dataset in the data folder
watershed = "watershed.shp"
print(f"pour_points: {pour_points}")
print(f"watershed: {watershed}")

# Store the Result object returned by Clip
points_clip_result = arcpy.analysis.Clip(
    pour_points, watershed, "pourpoints_clip.shp"
)
print(f"type(points_clip_result): {type(points_clip_result)}")

# getOutput(0) returns the first output, the output feature class path
points_clip_output = points_clip_result.getOutput(0)
print(f"points_clip_output: {points_clip_output}")
print(f"arcpy.Exists(points_clip_output): {arcpy.Exists(points_clip_output)}")

Clip only accepts certain combinations of geometry types. Point features can be clipped by points, lines, or polygons, but polygon features can only be clipped by polygons. When a tool fails, it raises an exception instead of returning a Result object, and the rest of the cell does not run.

In [ ]:
# Example 2 - Clip a polygon layer with a point layer
# This cell raises an exception on purpose

# We will use variables watershed and pour_points from Example 1
polygon_clip_result = arcpy.analysis.Clip(
    watershed, pour_points, "watershed_clip.shp"
)
print(f"type(polygon_clip_result): {type(polygon_clip_result)}")


The traceback above ends with the tool's error message. Because the tool failed, `polygon_clip_result` was never created. `arcpy.GetMessages()`, an ArcPy function, returns the messages from the last tool that ran, whether it succeeded or failed. It takes the same severity argument as `getMessages()`.

In [ ]:
# polygon_clip_result was never created, so this raises an error
# print(polygon_clip_result)
# NameError: name 'polygon_clip_result' is not defined

# GetMessages() returns the messages from the last tool that ran
print(arcpy.GetMessages())

# 2 returns only the error messages
print(f"Errors: '{arcpy.GetMessages(2)}'")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*